# Syn Bank Share of Wallet Intelligence Engine

## 1. Installing dependencies using "requirements.txt"

In [2]:
import sys
!{sys.executable} -m pip install -r requirements.txt

In [4]:
!{sys.executable} scripts/check_requirements.py

import pandas as pd
import numpy as np
import csv
import json
from pathlib import Path
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import joblib
from IPython.display import display

Checking project requirements...

[OK] pandas: 3.0.0
[OK] numpy: 2.3.3
[OK] matplotlib: 3.10.8
[OK] jupyterlab: 4.5.4
[OK] ipykernel: 6.29.5
[OK] nbformat: 5.10.4
[OK] pyspark: 4.2.0
[OK] pymupdf: 1.28.2
[OK] scikit-learn: 1.9.0
[OK] streamlit: 1.61.1
[OK] plotly: 6.9.0
[OK] anthropic: 0.86.0

All requirements are satisfied.


## 2. Important file requirements before running any code

### Required Raw Data Files

Before running the notebook, ensure the following raw data files are present in the **project root directory** with these exact filenames:

- `trade_finance.csv`
- `transactional_banking.csv`
- `cross_border_payments.csv`

> **Important:** The filenames must match exactly for the data-loading pipeline to run successfully.

## 3. Extract data from each client's external financial reports

> **Important:** The initial text extraction from the financial report PDFs has already been completed. Due to the large number and
> size of these reports, the pre-extracted output is provided directly. This avoids the need to load all financial reports and
> rerun the extraction process, significantly reducing setup time and computational overhead.


After extracting text from each pdf, an AI agent is fed the financial reports and produces "multi_year_figures.json", an overview of the information it was able to extract from each financial report. Next, "merge_multiyear.py" is run below to combine the AI agent's insights as well as the extracted text from each financial report, and produce a csv file summarising the financial information of each client over multiple financial years.

In [5]:
!{sys.executable} scripts/merge_multiyear.py

# Visualising the output produced by the above script
all_financials_df = pd.read_csv("hackathon-finreports/_extracted/financials_multiyear.csv")
all_financials_df.head(5)

Wrote 43 company-year rows to D:\Root3\hackathon-finreports\_extracted\financials_multiyear.csv
Wrote ML-ready (numeric-only) version to D:\Root3\hackathon-finreports\_extracted\financials_multiyear_ml.csv


,entity_name,canonical,fiscal_year,source_file,currency,fx_rate_to_zar,fx_rate_date,revenue_m,revenue_zar_m,cost_of_sales_m,...,trade_payables_m,trade_payables_zar_m,inventory_m,inventory_zar_m,foreign_revenue_pct,fx_gains_losses_m,fx_gains_losses_zar_m,fiscal_year_end,page_ref,notes
0,Anglo American,angloamerican,2024,aa-annual-report-full-2024.pdf,USD,16.3213,2026-08-07,27290.0,445408.3,NaN,...,6092.0,99429.4,6439.0,105092.9,NaN,-42.0,-685.5,31 December 2024,"236, 237, 240, 244, 267, 268","No single 'cost of sales' line disclosed, so a..."
1,Anglo American,angloamerican,2025,AA annual-report-full-2025.pdf,USD,16.3213,2026-08-07,18546.0,302694.8,NaN,...,4956.0,80888.4,3819.0,62331.0,NaN,-77.0,-1256.7,31 December 2025,Consolidated income statement; Consolidated ba...,No single cost-of-sales line; operating_expens...
2,AngloGold Ashanti,angloashanti,2024,AGA-AR24.pdf,USD,16.3213,2026-08-07,5793.0,94549.3,3726.0,...,957.0,15619.5,1213.0,19797.7,NaN,-87.0,-1420.0,31 December 2024,"148, 150, 156-157, 195, 205",fx_gains_losses (-87) is a combined 'Foreign e...
3,AngloGold Ashanti,angloashanti,2025,ERDEC25.pdf,USD,16.3213,2026-08-07,9893.0,161466.6,5022.0,...,1015.0,16566.1,1251.0,20417.9,NaN,-41.0,-669.2,31 December 2025,2025 earnings release financial statements,Revenue uses 'Revenue from product sales'. cos...
4,Aspen Pharmacare,aspen,2024,FY-2024-Annual-Results-Presentation.pdf,ZAR,1.0000,NaN,44706.0,44706.0,25252.0,...,10347.0,10347.0,18002.0,18002.0,NaN,-64.0,-64.0,30 June 2024,"6, 17, 18, 37, 42, 48","Presentation deck (results announcement), not ..."


> ### Updating the Engine with New Syn Bank Data
>
> To run the engine on updated or new client data:
>
> 1. Replace the three raw Syn Bank CSV files in the project root:
>    - `trade_finance.csv`
>    - `transactional_banking.csv`
>    - `cross_border_payments.csv`
>
> 2. If the data contains a **new client not already present in the extracted financial-report metadata**, add that client's fiscal-year information to:
>    - `client_fiscal_years.csv`
>
>    Required columns:
>
>    `entity_name, fiscal_year, fiscal_year_end`
>
> 3. Rerun **Step 4** to rebuild the Silver and Gold layers.
> 4. Rerun **Step 6** to generate updated ElasticNet wallet predictions.
> 5. Rerun **Step 7** to launch the Streamlit dashboard and GenAI layer using the updated portfolio.
>
> **Steps 1–3 and 5 do not need to be rerun unless the environment, external financial-report data, or trained models themselves need to be updated.**


## 4. Prepare the internal SynBank data - Run the medallion pipeline

The medallion pipeline has three main layers: bronze, silver and gold. In the context of this project, the bronze layer consists of the raw data, the silver layer consists of the cleaned and deduped data and the gold layer consists of the data that will be fed into the ML model to train it (and to later use for prediction when new data arrives)

> **For future data updates:** After replacing the three raw CSV files in the project root, rerun this step before proceeding directly to Steps 6 and 7.

In [ ]:
# Take the raw data and clean and dedupe it
!{sys.executable} medallion-pipeline/scripts/bronze_silver_transformation.py

# Prepare the cleaned data into the correct format to be fed into the ElasticNet ML model
!{sys.executable} medallion-pipeline/scripts/silver_gold_transformation.py

# Display the gold layer with the important features extracted
cross_border = pd.read_csv("medallion-pipeline/gold/cross_border_gold.csv")
transactional = pd.read_csv("medallion-pipeline/gold/transactional_banking_gold.csv")
trade = pd.read_csv("medallion-pipeline/gold/trade_finance_gold.csv")

display(cross_border.head(5))
display(transactional.head(5))
display(trade.head(5))

## 5. Combine Internal & External Data → Train the ElasticNet ML Models

The cleaned **internal Syn Bank data** is combined with the extracted **external financial data** by client and financial year to create the final modelling dataset.

#### Model Inputs and Targets

The **inputs (features)** consist of a combination of internal Syn Bank indicators and external company financial metrics. The **targets** represent the estimated total client wallet components that the models aim to predict.

Rather than training a single model, **five separate ElasticNet models** are trained for the different target variables. This allows each model to make the best use of the features that are most relevant to its specific target, rather than forcing all wallet components into a single modelling relationship.

#### Model Training & Validation

Due to the relatively small number of client-year observations, **Leave-One-Out Cross-Validation (LOOCV)** is used during model development. Each observation is held out once for validation while the model is trained on all remaining observations, allowing the available data to be used as efficiently as possible.

LOOCV is also used to **fine-tune the ElasticNet hyperparameters (`alpha` and `l1_ratio`)** for each of the five models. This allows the balance between L1 (Lasso) and L2 (Ridge) regularisation to be optimised independently for each target.

The final result is **five independently tuned ElasticNet models**, each designed to estimate a specific component of the client's total wallet from the available internal and external features.

In [ ]:
!{sys.executable} machine_learning/elastic_net.py




## 6. Feed data to the trained ML model and make predictions
This code runs "predict_wallet.py" which uses the ElasticNet trained models to make three statistical predictions: Synbank's share of a client's wallet, an estimate of the client's total wallet, and the calculated gap between SynBank's share of the wallet and the estimated total wallet. These insights are used and visualised by the GenAI layer in the next step.

> **Important:** The columns "predicted_total_wallet_zar_m" and "predicted_gap_zar_m" represent monetary amounts in MILLIONS

In [ ]:
from machine_learning.predict_wallet import MLWalletPredictor

predictor = MLWalletPredictor()

predictions_df = predictor.predict_all_clients()

display_df = predictions_df.copy()

display_df["predicted_share"] = display_df["predicted_share_pct"].apply(
    lambda x: "Not reliably estimable"
    if x <= 0
    else f"{x:.2f}%"
)

display(
    display_df[
        [
            "entity_name",
            "pillar",
            "target",
            "internal_zar",
            "predicted_share",
            "predicted_total_wallet_zar_m",
            "predicted_gap_zar_m",
        ]
    ]
)

## 7. Run the GenAI Layer - Launch the interactive dashboard

In [ ]:
!{sys.executable} -m streamlit cache clear
!{sys.executable} -m streamlit run dashboard/app.py

> **New-data workflow:** After replacing the raw Syn Bank CSV files, rerun **Steps 4, 6 and 7 only**.  
> For completely new clients, also provide their fiscal-year metadata in `client_fiscal_years.csv`.